In [ ]:
# Import dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Import our Behavior Agent
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
from Behavior_Agent.behavior_agent import BehaviorAgent
from Behavior_Agent.mock_data_generator import generate_all

print('Behavior Agent loaded successfully!')

## 1. Generate Mock Data
Since we don't have the real dataset yet, we generate simulated UEBA events matching the official data dictionary.

In [ ]:
# Generate mock data
ueba_df, host_df = generate_all(output_dir='.')

print(f'\nUEBA events: {len(ueba_df):,}')
print(f'Host profiles: {len(host_df):,}')
print(f'\nColumns: {list(ueba_df.columns)}')

In [ ]:
# Explore the data
print('=== Event Type Distribution ===')
print(ueba_df['event_type'].value_counts())
print(f'\n=== Attack Rate ===')
print(f"Normal: {(~ueba_df['is_anomaly']).sum():,}")
print(f"Anomaly: {ueba_df['is_anomaly'].sum():,}")
print(f"Rate: {ueba_df['is_anomaly'].mean():.1%}")

## 2. Initialize and Train the Behavior Agent

In [ ]:
# Initialize the agent
agent = BehaviorAgent(
    contamination=0.02,   # ~2% expected anomaly rate
    n_estimators=200,     # 200 trees in the Isolation Forest
    random_state=42
)

# Load host context (segments, criticality, honeypots)
agent.load_host_context(host_df)

# Train the Isolation Forest model
agent.fit(ueba_df)

print('\nAgent trained and ready!')

## 3. Score All Events (Hybrid: ML + Rules)

In [ ]:
# Score all events
results = agent.score_batch(ueba_df)

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)
results_df.head(10)

## 4. Visualize the Results

In [ ]:
# Score Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Overall score distribution
axes[0].hist(results_df['score'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0.65, color='orange', linestyle='--', label='Alert Threshold (0.65)')
axes[0].axvline(x=0.90, color='red', linestyle='--', label='Critical Threshold (0.90)')
axes[0].set_xlabel('Behavior Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution (All Events)')
axes[0].legend()

# Plot 2: ML Score vs Rule Score
colors = ['green' if s < 0.65 else 'orange' if s < 0.90 else 'red' for s in results_df['score']]
axes[1].scatter(results_df['ml_score'], results_df['rule_score'], 
                c=colors, alpha=0.5, s=10)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='ML = Rules')
axes[1].set_xlabel('ML Score (Isolation Forest)')
axes[1].set_ylabel('Rule Score (Security Guard)')
axes[1].set_title('ML Brain vs Security Guard')
axes[1].legend()

# Plot 3: Scores by ground truth
# Merge with original data to get ground truth
merged = results_df.copy()
merged['is_anomaly'] = ueba_df['is_anomaly'].values
normal_scores = merged[~merged['is_anomaly']]['score']
attack_scores = merged[merged['is_anomaly']]['score']
axes[2].hist(normal_scores, bins=30, alpha=0.6, label=f'Normal (n={len(normal_scores)})', color='green')
axes[2].hist(attack_scores, bins=30, alpha=0.6, label=f'Attack (n={len(attack_scores)})', color='red')
axes[2].set_xlabel('Behavior Score')
axes[2].set_ylabel('Count')
axes[2].set_title('Score Separation: Normal vs Attack')
axes[2].legend()

plt.tight_layout()
plt.savefig('behavior_agent_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: behavior_agent_scores.png')

## 5. Analyze the Saturday 3:18 AM Attack Scenario

In [ ]:
# Filter for the Saturday 3 AM attack events
attack_results = results_df[results_df['username'] == 'ram.shrestha'].copy()
attack_results = attack_results.sort_values('timestamp')

print('=== Saturday 3:18 AM Attack Detection ===')
print(f'Events from ram.shrestha: {len(attack_results)}')
print(f'Mean score: {attack_results["score"].mean():.4f}')
print(f'Max score:  {attack_results["score"].max():.4f}')
print(f'Min score:  {attack_results["score"].min():.4f}')
print(f'\nFlags triggered:')
all_flags = []
for flags in attack_results['flags']:
    if flags:
        all_flags.extend(flags)
for flag, count in pd.Series(all_flags).value_counts().items():
    print(f'  {flag}: {count}')

print(f'\nAll events detected as anomalous (score >= 0.65): '
      f'{(attack_results["score"] >= 0.65).all()}')

## 6. Generate Output for Correlation Agent

In [ ]:
# Aggregate by source IP
output_df = agent.aggregate_by_ip(results)

print('=== behavior_agent_output.csv ===')
print(f'Total IPs: {len(output_df)}')
print(f'Anomalous IPs: {(output_df["is_behavior_anomaly"] == 1).sum()}')
print(f'\nTop 10 Most Anomalous IPs:')
output_df.head(10)

In [ ]:
# Save the output
output_path = os.path.join('..', 'behavior_agent_output.csv')
output_df.to_csv(output_path, index=False)
print(f'Output saved to: {output_path}')
print(f'Rows: {len(output_df)}')
print(f'\nColumns: {list(output_df.columns)}')
print(f'\nThis file is ready for the Correlation Agent to merge!')

## 7. Performance Metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Get ground truth and predictions
y_true = ueba_df['is_anomaly'].astype(int).values
y_scores = np.array([r['score'] for r in results])
y_pred = (y_scores >= 0.65).astype(int)

# Classification Report
print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print(f'=== Confusion Matrix ===')
print(f'                  Predicted Normal  Predicted Anomaly')
print(f'Actual Normal     {cm[0][0]:<18} {cm[0][1]}')
print(f'Actual Anomaly    {cm[1][0]:<18} {cm[1][1]}')

# AUROC
if len(set(y_true)) > 1:
    auroc = roc_auc_score(y_true, y_scores)
    print(f'\nAUROC: {auroc:.4f}')